<div style="padding: 20px; background: linear-gradient(90deg, #1A2980 0%, #26D0CE 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🤖 Module 8.1: Introduction to Agentic RAG</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">From Passive Pipelines to Autonomous Thinkers.</p>
</div>

---

## 1. The Paradigm Shift

Up until now, our RAG systems have been **Passive**.
We, the developers, wrote hardcoded Python scripts: *Retrieve exactly 3 docs -> Insert into prompt -> Generate answer*.

**Agentic RAG** is **Active**. We give the LLM tools (like a Vector Search tool, a Web Search tool, a Calculator) and say: *"Here is a goal. You figure out which tools to use, when to use them, and when you are finished."*

| Feature | Traditional RAG | Agentic RAG |
| :--- | :--- | :--- |
| **Flow** | Fixed linear pipeline (`A -> B -> C`) | Dynamic loops (`Think -> Act -> Observe`) |
| **Retrieval** | Always searches DB once | Searches multiple times, or not at all! |
| **Reasoning** | None | Uses ReAct (Reasoning + Acting) logic |
| **Tools** | Only Retriever | Any tool (Search, SQL, Python execution) |

## 2. A Conceptual Comparison
Let's look at how the architecture differs conceptually before we build the real thing.

### Course alignment and free-first stack

- Covers: Agentic RAG capabilities and when workflows need stateful decisions.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
from dataclasses import dataclass
import time

@dataclass
class TraditionalRAG:
    query: str
    chunks: list[str]
    answer: str

@dataclass
class AgenticRAG:
    query: str
    thoughts: list[str]
    actions: list[str]
    answer: str

def run_traditional_rag(query: str) -> TraditionalRAG:
    """Fixed: retrieve → generate."""
    return TraditionalRAG(
        query=query, 
        chunks=["Doc 1: BERT is an encoder", "Doc 2: GPT is a decoder"],
        answer="Based on the docs, BERT is an encoder and GPT is a decoder."
    )

def run_agentic_rag(query: str) -> AgenticRAG:
    """Dynamic: reason → act → observe → repeat."""
    thoughts = []
    actions = []
    
    thoughts.append("I need to compare BERT and GPT.")
    actions.append("[Tool: SearchDB] Query: 'What is BERT?'")
    
    thoughts.append("Okay, BERT is an encoder. Now I need to know about GPT.")
    actions.append("[Tool: SearchDB] Query: 'What is GPT?'")
    
    thoughts.append("GPT is a decoder. I have enough information to answer the user.")
    actions.append("[Tool: Generate Final Answer]")
    
    return AgenticRAG(
        query=query, 
        thoughts=thoughts, 
        actions=actions,
        answer="BERT relies on an encoder architecture, whereas GPT relies on a decoder architecture."
    )

query = "What are the key differences between BERT and GPT?"
print(f"USER: {query}\n")

print("─"*50)
print("⚙️ TRADITIONAL RAG (Passive)")
print("─"*50)
t_result = run_traditional_rag(query)
print(f"1. Blindly Retrieved: {t_result.chunks}")
print(f"2. Blindly Generated: {t_result.answer}\n")

print("─"*50)
print("🧠 AGENTIC RAG (Active ReAct Loop)")
print("─"*50)
a_result = run_agentic_rag(query)
for i in range(len(a_result.thoughts)):
    print(f"[Thought] → {a_result.thoughts[i]}")
    time.sleep(0.5)
    if i < len(a_result.actions):
        print(f"[Action]  → {a_result.actions[i]}\n")
        time.sleep(0.5)

print(f"✨ FINAL ANSWER: {a_result.answer}")